In [64]:
import pandas as pd
import numpy as np

nscl_df = pd.read_csv("data/ncsl_ai_data_base.csv")
df = nscl_df.fillna("")

In [65]:
# Make column names easy to access
df = df.rename(columns={
    "State": "state",
    "Bill Number": "bill_number",
    "Bill Title": "title",
    "Bill Summary": "summary",
    "Bill Status": "status",
    "Category": "category",
    "URL": "url",
    "Page Year": "page_year",
})

In [66]:
# Rows with no bill, no legislative session, or data
is_removed = (df.bill_number == "") | df.bill_number.str.match(r"No\b|Not\b")
df = df[~is_removed]

In [ ]:
import re
import unicodedata


def normalize_row_text(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u2010-\u2015\u2212]", "-", s)   # en/em/other dashes -> "-"
    return re.sub(r"\s+", " ", s).strip()


for c in ["state", "bill_number", "title", "status",
          "summary", "category"]:
    df[c] = df[c].map(normalize_row_text)

In [68]:
# Bill status
bill_status_map = {
    "Enacted": "Passed",
    "Failed": "Failed",
    "Failed - Adjourned": "Failed",
    "Adopted": "Passed",
    "Pending": "Pending",
    "Pending - Carryover": "Pending",
    "To Governor": "Pending",
    "Override Pending": "Pending",
    "Vetoed": "Vetoed"
}

df.status = df.status.apply(
    lambda s: s if s != 'To governor' else 'To Governor')
df["status_type"] = df.status.map(bill_status_map)

df["status_stale"] = (df.status_type == "Pending") & (
    df.page_year < df.page_year.max())
df["outcome_final"] = df.status_type != "Pending"

In [69]:
# Fix one special case for bill_number
df.loc[df.bill_number == "B24-558", "bill_number"] = "B 558"

In [70]:
bill_audience_map = {'A': 'Assembly Bill',
                     'ACR': 'Assembly Concurrent Resolution',
                     'AJR': 'Assembly Joint Resolution',
                     'AR': 'Assembly Resolution',
                     'B': 'Bill',
                     'BDR': 'Bill Draft Request',
                     'D': 'Draft Bill',
                     'H': 'House Bill',
                     'HCR': 'House Concurrent Resolution',
                     'HD': 'House Docket',
                     'HJM': 'House Joint Memorial',
                     'HJR': 'House Joint Resolution',
                     'HM': 'House Memorial',
                     'HR': 'House Resolution',
                     'HSB': 'House Study Bill',
                     'L': 'Lesislative Bill',
                     'LCR': 'Legislative Council Resolution',
                     'LR': 'Legislative Reference',
                     'S': 'Senate Bill',
                     'SCR': 'Senate Concurrent Resolution',
                     'SD': 'Senate Docket',
                     'SJR': 'Senate Joint Resolution',
                     'SM': 'Senate Memorial',
                     'SPB': 'Senate Bill',
                     'SR': 'Senate Resolution',
                     'SSB': 'Senate Study Bill',
                     'WS': 'Senate Bill'}

# bill_label keeps the raw text ("H 197"); bill_num becomes the integer bill number (cleaned below)
df = df.rename(columns={"bill_number": "bill_label"})
df[['chamber_prefix', 'bill_num']] = df.bill_label.str.split(
    ' ', n=1, expand=True)
df["chamber_name"] = df.chamber_prefix.map(bill_audience_map)

In [71]:
import base64
import json


def url_bill_id(u):
    if not isinstance(u, str) or not u:
        return None
    m = re.search(r"ID:bill:([A-Za-z0-9]+)",
                  u)                         # StateNet
    if m:
        return m.group(1)

    # California leginfo, e.g. CA202120220AB13
    m = re.search(r"bill_id=(\d{8}\d?[A-Z]+\d+)", u)
    if m:
        return "CA" + m.group(1)

    # Maine legislature site, e.g. ME130HP1199
    m = re.search(r"paper=([A-Z]+\d+).*snum=(\d+)", u)
    if m:
        return "ME" + m.group(2) + m.group(1)
    # LexisNexis (base64 JSON)
    if "lexisnexis.com/opentext/" in u:
        tok = u.split("/opentext/")[1].split(".")[0]
        try:
            return json.loads(base64.urlsafe_b64decode(tok + "=" * (-len(tok) % 4)))["id"]
        except Exception:
            return None
    return None


df["url_bill_id"] = df.url.map(url_bill_id)
session = pd.to_numeric(df.url_bill_id.str.extract(r"^[A-Z]{2}(\d{4})")[0])
df["session_year"] = session.fillna(df.page_year).astype(int)

In [72]:
# bill_label is the raw string ("1154a", "1270 , First Special Session", etc.)
df["is_special_session"] = (df.bill_label.str.contains("session", case=False)
                            .groupby(df.url_bill_id.fillna(df.index.to_series())).transform("any"))
df["bill_id"] = df.url_bill_id.fillna(
    df.state + "_" + df.session_year.astype(str) + "_" + df.chamber_prefix + "_" + df.bill_label.str.replace(" ", ""))

df = df.sort_values(["bill_id", "page_year"]).drop(
    columns=["url_bill_id", "url"])

# Every listing of every bill (status can change between pages) - exported at end
history = df.copy()

In [73]:
# Fill title/category gaps from the latest earlier listing of the same bill that has a value
for c in ["title", "category"]:
    latest = (df.assign(**{c: df[c].replace("", np.nan)})
                .dropna(subset=[c]).groupby("bill_id")[c].last())
    df[c] = df[c].replace("", np.nan).fillna(df.bill_id.map(latest))

# How many pages listed each bill, then keep the latest listing
df["num_pages_listed"] = df.groupby("bill_id").page_year.transform("size")
df = df.drop_duplicates("bill_id", keep="last")

In [74]:
# Clean bill numbers and create an integer type column
df.bill_num = df.bill_num.str.extract(r"^(\d+)")[0].astype(int)

In [ ]:
# Correct resolutions typed as bills
res_tag = r"^\((?:Senate |House )?Resolution\)\s*"
is_tagged = df.summary.str.contains(res_tag, na=False)
df.loc[is_tagged, "chamber_name"] = df.loc[is_tagged,
                                           "chamber_name"].str.replace("Bill", "Resolution")
df["summary"] = df.summary.str.replace(res_tag, "", regex=True)

In [76]:
import us

state_mapping = us.states.mapping('name', 'abbr')
state_mapping.pop("Virgin Islands")
state_mapping['U.S. Virgin Islands'] = 'VI'

df.state = df.state.map(state_mapping)
history["state"] = history.state.map(state_mapping)

In [77]:
categories = ['Appropriations', 'Audit', 'Child Pornography', 'Criminal Use', 'Cybersecurity', 'Education Use', 'Education/Training',
              'Effect on Labor/Employment', 'Elections', 'Energy', 'Government Use', 'Health Use', 'Housing', 'Impact Assessment',
              'Judicial Use', 'Notification', 'Oversight/Governance', 'Personhood', 'Private Right of Action', 'Private Sector',
              'Provenance', 'Responsible Use', 'Studies', 'Taxes']


def clean_categories(s):
    if pd.isna(s):
        return np.nan
    found = [c for c in categories if c.lower() in s.lower()]
    return ";".join(found) if found else np.nan

In [78]:
df.category = df.category.str.replace(",", ";")

In [79]:
# Make category into ";"-separated list of above types
category_text = df.category.str.split(";")
category_text = category_text.explode().str.strip()

df["category"] = df.category.map(clean_categories)

print("Bills w/no categories: ", df.category.isna().sum())

Bills w/no categories:  76


In [80]:
# Replace curly quotes in text columns
quotes = str.maketrans({"\u2018": "'", "\u2019": "'",
                       "\u201c": '"', "\u201d": '"'})
for c in ["title", "summary"]:
    df[c] = df[c].map(lambda s: s.translate(quotes)
                      if isinstance(s, str) else s)

In [ ]:
df = df.replace("", np.nan)

# Reorder column names before export
column_order = ["bill_id", "state", "session_year", "page_year", "num_pages_listed",
                "chamber_name", "chamber_prefix", "bill_num", "bill_label", "is_special_session",
                "title", "summary", "status", "status_type", "status_stale", "outcome_final", "category"]

df = df[column_order]

In [82]:
# Final checks - each bill is unique, session year is corrected and no null values
assert df.bill_id.is_unique
assert (df.session_year <= df.page_year).all()
assert df.state.notna().all() and df.bill_num.notna().all()

print(df.isnull().sum()[lambda s: s > 0])
# assert (df.isnull().sum().sum() == 0)

title       41
category    76
dtype: int64


In [83]:
# Export - Bill status history
history[["bill_id", "state", "page_year", "status", "status_type"]].to_csv(
    "data/bill_status_history.csv", index=False)

# Export - Cleaned dataset
df.to_csv("data/ncsl_ai_data_clean.csv", index=False)